---

# Clustering

In this exercise, we will detect compound events of natural hazards. These events are characterized by the co-occurrence of two extreme events at the same location and time (or small time lag). In the course of climate change, these compound natural hazards will occur more often. \

Examples are: 
* Spatial and temporal co-occurrence of extreme rainfall and a winter storm event (<-- part of todays  exercise )
* A heavy rainfall within short time after a prolonged drought period  -> the soil is solidified and cannot absorb the rain, leading to a high surface runoff (and potential flash floods, landslides etc.)
* A heatwave during a drought period, leading to increased fire risk and, for instance, high economic losses for agricultural sector or loss of cooling water for nuclear power plants etc.

In the past some compound events caused even higher costs to society than the sum of the costs caused by their single events. In other words, compound events can lead to more negative impacts than when their single events would occur independently from each other.\

For this exercise, we define compound events as the spatial and temporal overlap of two hazard events. In particular, we focus on the detection of compound haazrd events in UK between December 2013 and early January 2014. During this time several winter storms hit different regions of UK, causing far-reach impacts, like power blackout, severe interruption in transportation sector, and economic losses as many shops were closed shortly before Christmas. 


The exercise is partly based on the study by [Tilloy et al. 2022](https://doi.org/10.5194/esd-13-993-2022): *"A methodology for the spatiotemporal identification of compound hazards: wind and precipitation extremes in Great Britain (1979-2019)"*
<!-- They define compound hazards:\
*".. here as the intersecting area (AND) on which two (or more) hazards develop during the aggregated union of the time periods (OR) of the two hazard events. We believe this definition is the most relevant in terms of impacts as it accounts for potential cascading or compounding impacts in time and space of two (or more) hazards (e.g. flooding of a building caused by a destroyed roof and heavy precipitation)."* -->


In [ ]:
import os
from pathlib import Path
from datetime import timedelta
import collections

import numpy as np
import pandas as pd
import geopandas as gpd 
import xarray as xr
from hdbscan import HDBSCAN
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature




## Download the data
Download the zip file `04_data_clustering.zip` from this [folder](https://tubcloud.tu-berlin.de/s/ZX6LbyAQzC5i6RL).
Unzip the folder and place it in a folder called `./data` in the same directory of this exercise.


## Load and explore data



https://weather.metoffice.gov.uk/binaries/content/assets/metofficegovuk/pdf/weather/learn-about/uk-past-events/interesting/2013/winter-storms-december-2013-to-january-2014---met-office.pdf

Load the weather data (average wind speed, precipitation accumulation) for UK from the attached data folder. The variables "p0001" and "p0005" refer to the accumulation of precipitation or the wind speed averaged over the last hour or last 5 hours , respectively. For the exercise, we use only the "p0001" variable. \
The meteorological data is from the ERA5-Land hourly dataset (hosted by the  Climate Change Service, C3S). ERA-5 is a reanalysis dataset of common meterological variables for the past decades, it means that the data is synthetically created by running Global and Regional Climate Models on empirical observations of the past weather.

As second step, load the respective administrative boundaries of UK (NUTS level 1) as shapefile: https://gadm.org/download_country.html. 
Use the `geopandas` package for reading the shapefile.


In [ ]:
data_dir = Path('./data')

# read in netcdf
rain_data_hourly = xr.open_dataset(data_dir / "raindat_0919.nc", engine="netcdf4") 
wind_data_hourly = xr.open_dataset(data_dir / "windat_0919.nc", engine="netcdf4", decode_times=True) 


# country borders
gdf_admin_units = gpd.read_file(data_dir / "country_borders/gadm41_GBR_2.shp")
gdf_admin_units.plot()

For keeping the computation load rather low, we just focus on the winter half year 2013 / 2014 (1st Oct. - 31st March).
In addition, we for now just use the timestamps around noon (12UTC) and midnight (0 UTC). Also crop the spatial extent of the data to the size of UK (at least more or less) for quicker computation.\
Hint: You can find out the coordinates of UK by visiting (koordinaten_umrechner)[https://www.koordinaten-umrechner.de/decimal/51.000000,10.000000?karte=OpenStreetMap&zoom=8]



In [ ]:


rain_daily = rain_data_hourly.sel(time=(rain_data_hourly.time.dt.hour.isin([0, 6, 12, 18]))) # or : rain_data_hourly.where(rain_data_hourly.time.dt.hour == 12, drop=True)
rain_daily = rain_daily.sel(longitude=slice(-7.6, 1.6), latitude=slice(59.0, 50.0), time=slice("2013-12-01", "2014-01-07"))

wind_daily = wind_data_hourly.sel(time=(wind_data_hourly.time.dt.hour.isin([0, 6, 12, 18]))) 
wind_daily = wind_daily.sel(longitude=slice(-7.6, 1.6), latitude=slice(59.0, 50.0), time=slice("2013-12-01", "2014-01-07"))


In [ ]:
# os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
wind_daily

## Get only extreme rainfall events
1. Set unit for the precipitation accumulation (over 1 hour) to millimeters for readability
2. Get the 95th percentile for each pixel during our time period. In this way, (I) we collect only timestamps with extreme rainfall or wind. By taking the percentile per gird-cell, (II) we are taking into account the geographical characteristics of UK. For example, wind storms are usually stronger at Scotlands coast than on the east cost (because of the jet stream), however, also the east coast can experience heavy wind storms which are exceptional for the area but usually over smaller magnitude than in Scotland. The same applies for the rainfall.  
In case your kernel crashes: approximate the value by using a smaller temporal subset. This is not the best approach but good enough when you dont have an dedicated graphics card ;)
3. Then select only the pxiels with extreme rainfall and wind speed based on your defined percentile  


In [ ]:
# 1.
rain_daily.p0001[:] = rain_daily.p0001 * 1000  # convert m to mm

# 2.
### percentile for each pixel averaged over time
rain_95th = rain_daily.p0001.quantile(0.99, dim="time")
wind_95th = wind_daily.p0001.quantile(0.99, dim="time")
print(f"""
      Thresholds (95th percentile) span:
      - for extreme rainfall: {rain_95th.min().values.round(3)} - {rain_95th.max().values.round(3)} mm
      - for wind: {wind_95th.min().values.round(3)} - {wind_95th.max().values.round(3)} m/s """
    )

# 3.
rain_extreme = rain_daily.where(rain_daily.p0001 >= rain_95th, drop=True)
wind_extreme = wind_daily.where(wind_daily.p0001 >= wind_95th, drop=True)


Plot the distributions of both meteorological variables

In [ ]:
rain_extreme.p0001.plot()

In [ ]:
wind_extreme.p0001.plot()

### Overview plot

Create a map with the rain data for a certain timestamp and add the UK boundaries as layer on top of it. \
Print also the time information for each plot  - The days differ but why? 

In [ ]:

# Here we take a certain day
#  Test different timestamps here to see temporal shift between wind and precipitation extremes -->
timestamp_rain = "2013-12-24T00:00:00"  # <----
timestamp_wind = "2013-12-24T00:00:00"  # <---- 

                       
# Set up the plot
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

for ax, xr_var in zip([ax1, ax2], [rain_extreme.p0001.sel(time=timestamp_rain), wind_extreme.p0001.sel(time=timestamp_wind)]):
    ax.coastlines()
    ax.add_feature(cfeature.OCEAN)

    # Add gridlines with labels
    gridlines = ax.gridlines(draw_labels=True, linestyle="--", color="gray", alpha=0.7)
    gridlines.top_labels = False
    gridlines.right_labels = False
    gridlines.left_labels = True
    gridlines.bottom_labels = True

    # Adjust the padding and font size of the labels
    gridlines.xlabel_style = {"size": 10, "color": "black"}
    gridlines.ylabel_style = {"size": 10, "color": "black"}

    # Plot the yearly mean rainfall
    pc = xr_var.plot.pcolormesh(
        ax=ax, cmap="rainbow_r",
        cbar_kwargs={"orientation": "vertical", "pad": 0.06, "shrink": 0.5, "aspect": 30}
    )

    # Overlay the shapefile using geopandas
    gdf_admin_units.plot(ax=ax, transform=ccrs.PlateCarree(), edgecolor="black", facecolor="none", linewidth=1)


ax1.set_title(f"Total Rainfall [mm], {timestamp_rain}")
ax2.set_title(f"Wind Speed [m/s], {timestamp_wind}")
plt.show()


In the two maps we see that there is a spatial overlay between the extreme precipitation and the wind event at the same day and timestep.
However, in some cases an extreme wind can occur a few hours before a heavy rainfall starts. As we keep from the hourly dataset only the 4 timesteps per observation day, we might not be able to ee this time-lag in our analysis

### Clip hazard data to the size of UK
The current version of our data shows also the meterological variables over the sea, however, we are interested only on the impacts of compound events on the land.\
1. First, check the coordinate reference system (CRS) for the meteorological data and the administrative data, respectively. Change the CRS if needed.
2. Second, dissolve all administrative units of the geopandas DataFrame (i.e. our shapefile) to get only the boundaries of UK. Have a look at the dataframe and decide for which attribute you would use to dissolve all administrative units into one single unit.
3. Then, overlay the meteorological variables with the boundaries of UK. 


In [ ]:
# 1.
print("CRS of the xarray data before setting:", rain_extreme.rio.crs)
rain_extreme.rio.write_crs(4326, inplace=True)
wind_extreme.rio.write_crs(4326, inplace=True)

print("CRS of the shapefile:", gdf_admin_units.crs)
print("CRS of the xarray data after setting:", rain_extreme.rio.crs)

In [ ]:
# 2.
gdf_admin_units.tail(3)

gdf_country = gdf_admin_units.dissolve(by='COUNTRY')
gdf_country

In [ ]:
# 3. 
rain_extreme = rain_extreme.rio.clip(gdf_country.geometry.values, gdf_country.crs)
wind_extreme = wind_extreme.rio.clip(gdf_country.geometry.values, gdf_country.crs)

## Spatiotemporal clustering of extreme events

As a next step we perform the HDBSCAN clustering and plot the results


In [ ]:
## Flatten images from 3D -> 2D array of pixels  
rain_extreme_flat = rain_extreme.p0001[:].values.reshape(rain_extreme.p0001.shape[0], -1) # 2D shape: time, lon *lat
rain_extreme_flat

# print(np.nanmax(rain_extreme.p0001.values))
# np.nanmax(rain_extreme_flat)

NOTE: HDBSCAN.fit() takes the dataset only as 2D array with either only one column (:, 1) or one row (1, :) 

### Perform HDBSCAN clustering for rainfall extremes


In [ ]:
 
hdb = HDBSCAN(min_cluster_size=5)  

clusterer = hdb.fit(rain_extreme_flat.reshape(-1,1)) # temporal_mean[:])

print(clusterer.labels_.shape) 

print("Number of clusters found over entire time span:", len(np.unique(clusterer.labels_) )) 
print("Occurrences per cluster class:\n ", collections.Counter(clusterer.labels_))
rain_extreme_clst = clusterer.labels_.reshape(rain_extreme.p0001.shape) 

print(rain_extreme_clst.shape) # original shape

## write the clustered data back to xarray dataset 
rain_extreme["hdbscan_clusters"] = (("time", "latitude", "longitude"), rain_extreme_clst)
rain_extreme

In [ ]:
## Number of clusters for a single time step, D
# collections.Counter(rain_extreme.hdbscan_clusters[7, :,:].values.reshape(-1))

### Perform HDBSCAN clustering for wind extremes


In [ ]:
wind_extreme_flat = wind_extreme.p0001[:].values.reshape(wind_extreme.p0001.shape[0], -1) # 2D shape: time, lon *lat

In [ ]:
hdb = HDBSCAN(min_cluster_size=5)  

clusterer = hdb.fit(wind_extreme_flat.reshape(-1,1))

print(clusterer.labels_.shape) 

print("Number of clusters found over entire time span:", len(np.unique(clusterer.labels_) )) 
print("Occurences per cluster class:\n ", collections.Counter(clusterer.labels_))
wind_extreme_clst = clusterer.labels_.reshape(wind_extreme.p0001.shape) 

print(wind_extreme_clst.shape) # original shape

## write the clustered data back to xarray dataset 
wind_extreme["hdbscan_clusters"] = (("time", "latitude", "longitude"), wind_extreme_clst)
wind_extreme

Lets re-use the plot of the overview map from before, but this time with the variable for the cluster. The clustered values refer to the different intensities of the single events, at the same time similar wind/precipitation intensities are grouped togehter and noise (i.e. singel points outside the hazard zone are removed (check if this only for DBSCAN (eps:2.4, n=10)). By visualizing the clustered intensities we can see that for some areas high intensities of rainfal overlap with high intensities of windspeed

In [ ]:
## see cluster labels for a single time step
# collections.Counter(rain_extreme.hdbscan_clusters.sel(time=timestamp_rain).values.reshape(-1))


In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(nrows=2, ncols=2, figsize=(12, 10), subplot_kw={"projection": ccrs.PlateCarree()})

timestamp_rain = "2013-12-24T00:00:00"  # <----
timestamp_wind = "2013-12-24T00:00:00"  # <---- 

for ax, xr_var in zip(
    [ax1, ax2, ax3, ax4], 
    [rain_extreme.hdbscan_clusters.sel(time=timestamp_rain), rain_extreme.p0001.sel(time=timestamp_rain),
     wind_extreme.hdbscan_clusters.sel(time=timestamp_wind), wind_extreme.p0001.sel(time=timestamp_wind)]
    ):
    ax.coastlines()
    ax.add_feature(cfeature.OCEAN)

    # Add gridlines with labels
    gridlines = ax.gridlines(draw_labels=True, linestyle="--", color="gray", alpha=0.7)
    gridlines.top_labels = False
    gridlines.right_labels = False
    gridlines.left_labels = True
    gridlines.bottom_labels = True

    # Adjust the padding and font size of the labels
    gridlines.xlabel_style = {"size": 10, "color": "black"}
    gridlines.ylabel_style = {"size": 10, "color": "black"}

    # Plot the yearly mean rainfall
    pc = xr_var.plot.pcolormesh(
        ax=ax, cmap="rainbow_r",
        cbar_kwargs={"orientation": "vertical", "pad": 0.06, "shrink": 0.5, "aspect": 30}
    )

    # Overlay the shapefile using geopandas
    gdf_admin_units.plot(ax=ax, transform=ccrs.PlateCarree(), edgecolor="black", facecolor="none", linewidth=1)

ax1.set_title(f"Clustered Rainfall, \n {timestamp_rain}")
ax2.set_title(f"Accumulated Rainfall [mm], \n {timestamp_rain}")
ax3.set_title(f"Clustered Wind Speed, \n {timestamp_wind}")
ax4.set_title(f"Avg. Wind Speed [m/s], \n {timestamp_wind}")
plt.show()


## Extract the footprints of compound clusters
In this step, we spatiotemproal overlay the footprints of the extreme wind events with the footprints of the precipitation events.  \
This step is quite typical in environmental-research and often referred to as intersection. In the code block below perform a spatiotemoral  intersection by overlaying the footprints of the precipitation and wind events for the same timestamps, then keep only the footprint area where both hazards overlap. 

In addition, remove cluster values below or equal to 0, which allows us to better investigate the compound cluster footprints on the maps.

In [ ]:
cc_rain = rain_extreme.hdbscan_clusters.where(wind_extreme.hdbscan_clusters>0, drop=True).where(rain_extreme.hdbscan_clusters > 0, drop=True)
cc_wind = wind_extreme.hdbscan_clusters.where(rain_extreme.hdbscan_clusters>0, drop=True).where(wind_extreme.hdbscan_clusters > 0, drop=True)



In [ ]:
timestamp_init = "2013-12-22T00:00:00"  # <----


for additional_hours in range(0, 72, 6):  # next 3days

    timestamp_init = pd.Timestamp(timestamp_init)+ timedelta(hours=additional_hours)
    print("New timestamp hour: ", timestamp_init)

    fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(ncols=5, figsize=(24, 8), subplot_kw={"projection": ccrs.PlateCarree()})
    
    try: 
        for ax, xr_var in zip(
            [ax1, ax2, ax3, ax4, ax5], 
            [
                cc_rain.sel(time=timestamp_init), 
                # cc_wind.sel(time=timestamp_init), 
                rain_extreme.hdbscan_clusters.sel(time=timestamp_init), 
                wind_extreme.hdbscan_clusters.sel(time=timestamp_init),
                rain_extreme.p0001.sel(time=timestamp_init), 
                wind_extreme.p0001.sel(time=timestamp_init)]
        ):
                
            ax.coastlines()
            ax.add_feature(cfeature.OCEAN)

            # Add gridlines with labels
            gridlines = ax.gridlines(draw_labels=True, linestyle="--", color="gray", alpha=0.7)
            gridlines.top_labels = False
            gridlines.right_labels = False
            gridlines.left_labels = True
            gridlines.bottom_labels = True

            # Adjust the padding and font size of the labels
            gridlines.xlabel_style = {"size": 10, "color": "black"}
            gridlines.ylabel_style = {"size": 10, "color": "black"}

            # Plot the yearly mean rainfall
            pc = xr_var.plot.pcolormesh(
                ax=ax, cmap="rainbow_r",
                cbar_kwargs={"orientation": "vertical", "pad": 0.06, "shrink": 0.5, "aspect": 30}
            )
            

    except KeyError:
        print("Timestamp not found: ", timestamp_init)
        continue

    ax1.set_title(f"Clustered Compound (rain values), \n{timestamp_init}")
    ax2.set_title(f"Clustered Rainfall [mm], \n{timestamp_init}")
    ax3.set_title(f"Clustered Wind [m/s], \n{timestamp_init}")
    ax4.set_title(f"Total Rainfall [mm], \n{timestamp_init}")
    ax5.set_title(f"Total Wind Speed [m/s], \n{timestamp_init}")
    plt.show()


In the step before, we overlay the precipitation cluster footprints (regardless of their intensity strength / cluster value) with the clustered footprints for wind speed.

### Which regions in UK were most exposed to compound wind-precipitation events in winter 2013/2014?

Find out in which of the regions the majority of compound events happened. For this task, you can focus on small time periods, when single winter storms crossed UK. This happened, for example, around between 23rd - 24th December or around the 4th-6th December.

1. Simply count for each pixel the number of cluster values for the entire time period. Again we dont focus on the cluster values but rather on the sizes of the clusters.
2. Create a map showing the regions which were most exposed to compound hazard events
3. Compare the identified regions with the regions mentioned in a report from the Meterological Weather Service of UK (MetOffice): https://weather.metoffice.gov.uk/binaries/content/assets/metofficegovuk/pdf/weather/learn-about/uk-past-events/interesting/2013/winter-storms-december-2013-to-january-2014---met-office.pdf


In [ ]:
## 1.
# calculate mean over time for the compound event
# cc_wind_count_5th = cc_wind.sel(time=slice("2013-12-04T00:00:00", "2013-12-07T00:00:00")).count(dim="time") # southern England
cc_wind_count_23rd = cc_wind.sel(time=slice("2013-12-22T00:00:00", "2013-12-25T00:00:00")).count(dim="time") # southern England



In [ ]:
## 2. 23-25th Dec
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

ax.coastlines()
ax.add_feature(cfeature.OCEAN)

# Add gridlines with labels
gridlines = ax.gridlines(draw_labels=True, linestyle="--", color="gray", alpha=0.7)
gridlines.top_labels = False
gridlines.right_labels = False
gridlines.left_labels = True
gridlines.bottom_labels = True

# Adjust the padding and font size of the labels
gridlines.xlabel_style = {"size": 10, "color": "black"}
gridlines.ylabel_style = {"size": 10, "color": "black"}

pc = cc_wind_count_23rd.plot.pcolormesh(
    ax=ax, cmap="Reds",
)
gdf_admin_units.plot(ax=ax, transform=ccrs.PlateCarree(), edgecolor="black", facecolor="none", linewidth=1)
ax.set_title("Clustered Compound Event [Counted over time between 22nd-24th Dec]")

For the 23rd -24th time period: \
The map shows that most parts of southern UK experienced compound events. Some of them were even more exposed (i.e., all pixels with value 2), as they experienced either a single compound event lasting more than 6 hours or experienced several smaller compound events within only a few days.\
Use the MetOffice report and the internet to check whether the regions that experienced the most severe hazard impacts were also detected by our approach as compound events.\
Think about some reasons, why the map above shows also regions which experienced compound events, but were not mentioned as heavily impacted in the MetOffice report. 



In [ ]:

# Here we take a certain day
#  Test different timestamps here to see temporal shift between wind and precipitation extremes -->
timestamp_rain = "2013-12-24T00:00:00"  # <----
timestamp_wind = "2013-12-24T00:00:00"  # <---- 

                       
# Set up the plot
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(10, 8), subplot_kw={"projection": ccrs.PlateCarree()})

for ax, xr_var in zip([ax1, ax2], [rain_extreme.p0001.sel(time=timestamp_rain), wind_extreme.p0001.sel(time=timestamp_wind)]):
    ax.coastlines()
    ax.add_feature(cfeature.OCEAN)

    # Add gridlines with labels
    gridlines = ax.gridlines(draw_labels=True, linestyle="--", color="gray", alpha=0.7)
    gridlines.top_labels = False
    gridlines.right_labels = False
    gridlines.left_labels = True
    gridlines.bottom_labels = True

    # Adjust the padding and font size of the labels
    gridlines.xlabel_style = {"size": 10, "color": "black"}
    gridlines.ylabel_style = {"size": 10, "color": "black"}

    # Plot the yearly mean rainfall
    pc = xr_var.plot.pcolormesh(
        ax=ax, cmap="rainbow_r",
        cbar_kwargs={"orientation": "vertical", "pad": 0.06, "shrink": 0.5, "aspect": 30}
    )

    # Overlay the shapefile using geopandas
    gdf_admin_units.plot(ax=ax, transform=ccrs.PlateCarree(), edgecolor="black", facecolor="none", linewidth=1)


ax1.set_title(f"Total Rainfall [mm], {timestamp_rain}")
ax2.set_title(f"Wind Speed [m/s], {timestamp_wind}")
plt.show()


# Done! Have a nice rest of the day